In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# inference.py
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.nn.functional import softmax
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def format_input(prompt, response_a, response_b):
    return f"{prompt} </s> Response A: {response_a} </s> Response B: {response_b}"

def predict(model_dir, tokenizer_path, test_path, save_path, max_length=512):
    # Load tokenizer and model
    batch_size = 32
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE)
    model.eval()

    # Load test data
    df = pd.read_csv(test_path)

    # Pre-tokenize inputs for speed
    texts = [
        f"{row['prompt']} </s> Response A: {row['response_a']} </s> Response B: {row['response_b']}"
        for _, row in df.iterrows()
    ]
    ids = df["id"].tolist()

    # Batch inference
    all_probs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]
        batch_ids = ids[i:i+batch_size]

        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True,
                           truncation=True, max_length=max_length).to(DEVICE)

        with torch.no_grad():
            logits = model(**inputs).logits  # shape: (B, 3)
            probs = softmax(logits, dim=-1).cpu().numpy()

        all_probs.extend(probs)

    # Save submission
    pred_df = pd.DataFrame(all_probs, columns=["winner_model_a", "winner_model_b", "winner_tie"])
    pred_df["id"] = ids
    pred_df = pred_df[["id", "winner_model_a", "winner_model_b", "winner_tie"]]
    pred_df.to_csv(save_path, index=False)
    print(f"Saved submission to {save_path}")

In [ ]:
predict(
        model_dir="/kaggle/input/teacher-student-model/pytorch/default/1/student_model",
        tokenizer_path="/kaggle/input/teacher-student-model/pytorch/default/1/teacher_model",
        test_path="/kaggle/input/llm-classification-finetuning/test.csv",
        save_path="/kaggle/working/submission.csv"
    )